In [48]:
import numpy as np
import pandas as pd
import pickle  
import glob
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import random

import sys
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path


In [49]:
File_names = [
'MM_20220228_secondsite.pickle',
'MM_20220228_firstsite.pickle',
'MM_20211203.pickle',
'MM_20211123.pickle',
'MM_20230415.pickle',
'MM_20230414_firstsite.pickle',
'MM_20230413_secondsite.pickle',
'MM_20230413_firstsite.pickle',
'MM_20230228.pickle',
'MM_20230221.pickle',
'MM_20230216.pickle',
'MM_20220916.pickle',
'MM_20220621.pickle',
'MM_20220421.pickle',
'MM_20220408.pickle',
'MM_20220314.pickle',
]

In [50]:
# Inspect aux keys for each dataset listed in File_names
# Goal: check whether some files contain multiple experiments in data_local['aux'].

if 'File_names' not in globals():
    raise RuntimeError("Variable File_names is not defined in the current kernel.")

# Candidate roots used in this notebook/workspace
candidate_roots = []
for var_name in ['direct_path', 'module_path', 'data_file_path']:
    if var_name in globals() and isinstance(globals()[var_name], str):
        candidate_roots.append(globals()[var_name])
# Add common workspace fallbacks
candidate_roots.extend([
    os.getcwd(),
    '/Users/markusmeister/Work/Grants/MagnetSearch/Analysis MM',
    '/Users/markusmeister/Work/Grants/MagnetSearch/Github/MagnetSearch',
    '/Users/markusmeister/Work/Grants/MagnetSearch',
])
# Deduplicate while preserving order
seen = set()
roots = []
for r in candidate_roots:
    if r and r not in seen:
        seen.add(r)
        roots.append(r)


def resolve_candidates(fn):
    candidates = []

    if os.path.isabs(fn):
        candidates.append(fn)
    else:
        candidates.append(fn)
        for root in roots:
            candidates.append(os.path.join(root, fn))

    # Also search recursively under roots for exact basename matches
    base = os.path.basename(fn)
    for root in roots:
        root_path = Path(root)
        if root_path.exists() and root_path.is_dir():
            try:
                for p in root_path.rglob(base):
                    candidates.append(str(p))
            except Exception:
                pass

    # Try common extension toggles
    expanded = []
    for pth in candidates:
        expanded.append(pth)
        if pth.endswith('.pickle'):
            expanded.append(pth[:-7] + '.pkl')
        elif pth.endswith('.pkl'):
            expanded.append(pth[:-4] + '.pickle')

    # Unique preserve order
    uniq = []
    seen_local = set()
    for pth in expanded:
        if pth not in seen_local:
            seen_local.add(pth)
            uniq.append(pth)
    return uniq


def load_data_local(path_try):
    # Decide loader by extension first
    lower = path_try.lower()
    if lower.endswith('.pickle') or lower.endswith('.pkl'):
        with open(path_try, 'rb') as f:
            return pickle.load(f)

    # Fallback: npy-like object container
    obj = np.load(path_try, allow_pickle=True)
    if hasattr(obj, 'item'):
        try:
            return obj.item()
        except Exception:
            return obj
    return obj


file_aux_keys_summary = {}

print(f"Total entries in File_names: {len(File_names)}")
print("=" * 88)

for fn in File_names:
    uniq = resolve_candidates(fn)

    loaded = False
    last_err = None
    used_path = None
    data_local = None

    for path_try in uniq:
        if not os.path.exists(path_try):
            continue

        try:
            data_local = load_data_local(path_try)
            used_path = path_try
            loaded = True
            break
        except Exception as e:
            last_err = e

    print(f"File_names entry: {fn}")

    if not loaded:
        print("  status: could not load")
        if last_err is not None:
            print(f"  last error: {type(last_err).__name__}: {last_err}")
        print("  aux keys: <unavailable>")
        print("-" * 88)
        file_aux_keys_summary[fn] = {
            'loaded': False,
            'path_used': None,
            'aux_keys': None,
            'n_aux_keys': None,
            'multiple_experiments': None,
        }
        continue

    aux_keys = []
    if isinstance(data_local, dict) and 'aux' in data_local:
        aux_obj = data_local['aux']
        if hasattr(aux_obj, 'keys'):
            aux_keys = list(aux_obj.keys())

    n_aux = len(aux_keys)
    multi = n_aux > 1

    print(f"  loaded from: {used_path}")
    print(f"  n(aux keys): {n_aux}")
    print(f"  multiple experiments? {'YES' if multi else 'NO'}")
    print(f"  aux keys: {aux_keys}")
    print("-" * 88)

    file_aux_keys_summary[fn] = {
        'loaded': True,
        'path_used': used_path,
        'aux_keys': aux_keys,
        'n_aux_keys': n_aux,
        'multiple_experiments': multi,
    }

print("Done. Summary dict saved in file_aux_keys_summary")

Total entries in File_names: 16
File_names entry: MM_20220228_secondsite.pickle
  loaded from: c:\Users\dan\Documents\magnet_search\data\MM_20220228_secondsite.pickle
  n(aux keys): 4
  multiple experiments? YES
  aux keys: ['2nd site 2 Hz_2022-02-28_17-48-17', '2nd site 5 Hz_2022-02-28_17-50-52', '2nd site 10 Hz_2022-02-28_17-53-18', '2nd site 1 Hz_2022-02-28_17-54-46']
----------------------------------------------------------------------------------------
File_names entry: MM_20220228_firstsite.pickle
  loaded from: c:\Users\dan\Documents\magnet_search\data\MM_20220228_firstsite.pickle
  n(aux keys): 3
  multiple experiments? YES
  aux keys: ['5Hz_2022-02-28_16-06-03', '10 Hz_2022-02-28_16-21-40', '2 Hz_2022-02-28_16-25-47']
----------------------------------------------------------------------------------------
File_names entry: MM_20211203.pickle
  status: could not load
  aux keys: <unavailable>
-------------------------------------------------------------------------------------

In [51]:
# Compact summary of aux keys from file_aux_keys_summary
if 'file_aux_keys_summary' not in globals():
    raise RuntimeError('file_aux_keys_summary not found. Run the inspection cell first.')

print('Per-file aux key summary')
print('=' * 88)

n_loaded = 0
n_multi = 0

for fn, info in file_aux_keys_summary.items():
    loaded = info.get('loaded', False)
    if loaded:
        n_loaded += 1
        n_aux = info.get('n_aux_keys', 0)
        aux_keys = info.get('aux_keys', [])
        multi = bool(info.get('multiple_experiments', False))
        n_multi += int(multi)
        print(f"{fn}: n_aux={n_aux}, multiple_experiments={'YES' if multi else 'NO'}")
        print(f"  aux keys: {aux_keys}")
    else:
        print(f"{fn}: NOT LOADED")

print('-' * 88)
print(f"Loaded {n_loaded}/{len(file_aux_keys_summary)} files")
print(f"Files with multiple experiments (n_aux > 1): {n_multi}")

Per-file aux key summary
MM_20220228_secondsite.pickle: n_aux=4, multiple_experiments=YES
  aux keys: ['2nd site 2 Hz_2022-02-28_17-48-17', '2nd site 5 Hz_2022-02-28_17-50-52', '2nd site 10 Hz_2022-02-28_17-53-18', '2nd site 1 Hz_2022-02-28_17-54-46']
MM_20220228_firstsite.pickle: n_aux=3, multiple_experiments=YES
  aux keys: ['5Hz_2022-02-28_16-06-03', '10 Hz_2022-02-28_16-21-40', '2 Hz_2022-02-28_16-25-47']
MM_20211203.pickle: NOT LOADED
MM_20211123.pickle: NOT LOADED
MM_20230415.pickle: n_aux=23, multiple_experiments=YES
  aux keys: ['2023-04-15_15-56-12_W1R_mag2', '2023-04-15_16-06-20_W1R_mag2_inclined', '2023-04-15_15-57-55_W1R_mag3', '2023-04-15_16-03-43_W1R_mag3_inclined', '2023-04-15_15-59-40_W1R_mag8', '2023-04-15_16-01-59_W1R_mag8_inclined', '2023-04-15_16-37-23_W1R_3D_WN_Samechan', '2023-04-15_16-08-19_W1R_visual_3Hz0', '2023-04-15_16-08-19_W1R_visual_3Hz45', '2023-04-15_16-08-19_W1R_visual_3Hz270', '2023-04-15_16-08-19_W1R_visual_3Hz90', '2023-04-15_16-08-19_W1R_visual_3Hz2

In [52]:
# Tiny summary only: counts and which files have multiple experiments
rows = []
for fn, info in file_aux_keys_summary.items():
    rows.append((fn, info.get('loaded', False), info.get('n_aux_keys', None), info.get('multiple_experiments', None)))

n_total = len(rows)
n_loaded = sum(int(r[1]) for r in rows)
n_multi = sum(int(bool(r[3])) for r in rows if r[1])

print(f'Total files: {n_total}')
print(f'Loaded files: {n_loaded}')
print(f'Files with multiple experiments (n_aux > 1): {n_multi}')
print('---')
for fn, loaded, n_aux, multi in rows:
    print(f'{fn} | loaded={loaded} | n_aux={n_aux} | multiple={multi}')

Total files: 16
Loaded files: 14
Files with multiple experiments (n_aux > 1): 13
---
MM_20220228_secondsite.pickle | loaded=True | n_aux=4 | multiple=True
MM_20220228_firstsite.pickle | loaded=True | n_aux=3 | multiple=True
MM_20211203.pickle | loaded=False | n_aux=None | multiple=None
MM_20211123.pickle | loaded=False | n_aux=None | multiple=None
MM_20230415.pickle | loaded=True | n_aux=23 | multiple=True
MM_20230414_firstsite.pickle | loaded=True | n_aux=28 | multiple=True
MM_20230413_secondsite.pickle | loaded=True | n_aux=25 | multiple=True
MM_20230413_firstsite.pickle | loaded=True | n_aux=27 | multiple=True
MM_20230228.pickle | loaded=True | n_aux=7 | multiple=True
MM_20230221.pickle | loaded=True | n_aux=10 | multiple=True
MM_20230216.pickle | loaded=True | n_aux=2 | multiple=True
MM_20220916.pickle | loaded=True | n_aux=14 | multiple=True
MM_20220621.pickle | loaded=True | n_aux=4 | multiple=True
MM_20220421.pickle | loaded=True | n_aux=5 | multiple=True
MM_20220408.pickle | lo

In [53]:
# Diagram full structure for MM_20220408.pickle (without spike-array contents)
import os
import pickle
import numpy as np

TARGET_FILE = 'MM_20220408.pickle'

# Resolve path from prior summary if available
path_candidates = []
if 'file_aux_keys_summary' in globals() and TARGET_FILE in file_aux_keys_summary:
    p0 = file_aux_keys_summary[TARGET_FILE].get('path_used', None)
    if p0:
        path_candidates.append(p0)
        
path_candidates = list(dict.fromkeys(path_candidates))

target_path = None
for p in path_candidates:
    if os.path.exists(p):
        target_path = p
        break

if target_path is None:
    raise FileNotFoundError(f'Could not find {TARGET_FILE}')

with open(target_path, 'rb') as f:
    data_local = pickle.load(f)


def is_spike_name(name):
    s = str(name).lower()
    return ('spike' in s) or ('spikes' in s)


def array_summary(arr):
    try:
        return f'ndarray shape={arr.shape}, dtype={arr.dtype}'
    except Exception:
        return 'ndarray'


def walk(obj, name='root', prefix='', is_last=True, depth=0):
    connector = '└── ' if is_last else '├── '
    line_prefix = prefix + connector
    child_prefix = prefix + ('    ' if is_last else '│   ')

    # Dict-like
    if isinstance(obj, dict):
        print(f"{line_prefix}{name}/ [dict, {len(obj)} keys]")
        keys = list(obj.keys())
        for i, k in enumerate(keys):
            v = obj[k]
            last = (i == len(keys) - 1)
            # For spike-named keys: never print raw array contents
            if is_spike_name(k) and isinstance(v, np.ndarray):
                c = '└── ' if last else '├── '
                print(f"{child_prefix}{c}{k}: {array_summary(v)} [contents omitted]")
            else:
                walk(v, name=str(k), prefix=child_prefix, is_last=last, depth=depth + 1)
        return

    # List / tuple
    if isinstance(obj, (list, tuple)):
        tname = 'list' if isinstance(obj, list) else 'tuple'
        print(f"{line_prefix}{name}/ [{tname}, len={len(obj)}]")
        for i, v in enumerate(obj):
            last = (i == len(obj) - 1)
            walk(v, name=f'[{i}]', prefix=child_prefix, is_last=last, depth=depth + 1)
        return

    # NumPy arrays: show metadata only
    if isinstance(obj, np.ndarray):
        if is_spike_name(name):
            print(f"{line_prefix}{name}: {array_summary(obj)} [contents omitted]")
        else:
            print(f"{line_prefix}{name}: {array_summary(obj)}")
        return

    # Scalars / strings / other objects
    t = type(obj).__name__
    if isinstance(obj, (int, float, bool, np.integer, np.floating, np.bool_)):
        print(f"{line_prefix}{name}: {obj} [{t}]")
    elif isinstance(obj, str):
        s = obj if len(obj) <= 80 else (obj[:77] + '...')
        print(f"{line_prefix}{name}: '{s}' [str]")
    else:
        print(f"{line_prefix}{name}: [{t}]")


print(f"Loaded: {target_path}")
print('Tree diagram:')
print('root/ [loaded object]')
walk(data_local, name='data_local', prefix='', is_last=True)
print('\nNote: spike array contents are omitted; arrays are shown as shape/dtype only.')

Loaded: c:\Users\dan\Documents\magnet_search\data\MM_20220408.pickle
Tree diagram:
root/ [loaded object]
└── data_local/ [dict, 2 keys]
    ├── spikes/ [dict, 560 keys]
    │   ├── 1: ndarray shape=(8709,), dtype=float64
    │   ├── 4: ndarray shape=(1565,), dtype=float64
    │   ├── 5: ndarray shape=(8423,), dtype=float64
    │   ├── 7: ndarray shape=(5374,), dtype=float64
    │   ├── 8: ndarray shape=(30143,), dtype=float64
    │   ├── 9: ndarray shape=(79,), dtype=float64
    │   ├── 11: ndarray shape=(7158,), dtype=float64
    │   ├── 12: ndarray shape=(33549,), dtype=float64
    │   ├── 13: ndarray shape=(44855,), dtype=float64
    │   ├── 17: ndarray shape=(42,), dtype=float64
    │   ├── 19: ndarray shape=(84,), dtype=float64
    │   ├── 20: ndarray shape=(501,), dtype=float64
    │   ├── 22: ndarray shape=(1870,), dtype=float64
    │   ├── 23: ndarray shape=(27338,), dtype=float64
    │   ├── 24: ndarray shape=(109,), dtype=float64
    │   ├── 25: ndarray shape=(9,), dtype=floa

In [54]:
# MM_20220408: print aux keys+values and max spike time across spikes branch
import os
import pickle
import numpy as np

TARGET_FILE = 'MM_20220408.pickle'

# Reuse loaded object if available and appears to match target
use_existing = isinstance(globals().get('data_local', None), dict) and ('aux' in data_local) and ('spikes' in data_local)

if not use_existing:
    path_candidates = []
    if 'file_aux_keys_summary' in globals() and TARGET_FILE in file_aux_keys_summary:
        p0 = file_aux_keys_summary[TARGET_FILE].get('path_used', None)
        if p0:
            path_candidates.append(p0)

    for root in [
        os.getcwd(),
        '/Users/markusmeister/Work/Grants/MagnetSearch/Analysis MM',
        '/Users/markusmeister/Work/Grants/MagnetSearch/Github/MagnetSearch',
        '/Users/markusmeister/Work/Grants/MagnetSearch',
    ]:
        path_candidates.append(os.path.join(root, TARGET_FILE))

    path_candidates = list(dict.fromkeys(path_candidates))
    target_path = next((p for p in path_candidates if os.path.exists(p)), None)
    if target_path is None:
        raise FileNotFoundError(f'Could not find {TARGET_FILE}')

    with open(target_path, 'rb') as f:
        data_local = pickle.load(f)


def format_value(v, max_items=10):
    if isinstance(v, np.ndarray):
        if v.size <= max_items:
            return f"ndarray(shape={v.shape}, dtype={v.dtype}, values={v.tolist()})"
        return f"ndarray(shape={v.shape}, dtype={v.dtype}, first={v.flat[:max_items].tolist()}, ... )"
    if isinstance(v, dict):
        return f"dict({len(v)} keys)"
    if isinstance(v, (list, tuple)):
        if len(v) <= max_items and all(not isinstance(x, (dict, list, tuple, np.ndarray)) for x in v):
            return f"{type(v).__name__}({list(v)})"
        return f"{type(v).__name__}(len={len(v)})"
    s = repr(v)
    return s if len(s) <= 200 else (s[:197] + '...')


print('AUX branch: keys and values')
print('=' * 88)
aux = data_local.get('aux', None)
if not isinstance(aux, dict):
    print('aux is missing or not a dict.')
else:
    for k, v in aux.items():
        print(f"{k}:")
        if isinstance(v, (tuple, list)):
            for i, elem in enumerate(v):
                print(f"  [{i}] {format_value(elem)}")
        else:
            print(f"  {format_value(v)}")


def iter_numeric_arrays(obj):
    if isinstance(obj, np.ndarray):
        if obj.size > 0 and np.issubdtype(obj.dtype, np.number):
            yield obj
        return
    if isinstance(obj, dict):
        for v in obj.values():
            yield from iter_numeric_arrays(v)
        return
    if isinstance(obj, (list, tuple)):
        for v in obj:
            yield from iter_numeric_arrays(v)
        return


spikes_branch = data_local.get('spikes', None)
max_spike_time = None
n_arrays_scanned = 0

if spikes_branch is None:
    print('\nspikes branch is missing.')
else:
    for arr in iter_numeric_arrays(spikes_branch):
        n_arrays_scanned += 1
        try:
            local_max = np.nanmax(arr)
            if np.isfinite(local_max):
                if (max_spike_time is None) or (local_max > max_spike_time):
                    max_spike_time = float(local_max)
        except Exception:
            pass

    print('\nSPIKES branch summary')
    print('=' * 88)
    print(f'Numeric arrays scanned: {n_arrays_scanned}')
    print(f'Largest spike time across spikes branch: {max_spike_time}')

AUX branch: keys and values
mag10hz_2022-04-08_15-26-22:
  [0] ndarray(shape=(2,), dtype=float64, values=[1403.7345666666668, 1480.3381666666667])
  [1] 10
mag5hz_2022-04-08_15-28-47:
  [0] ndarray(shape=(2,), dtype=float64, values=[1516.1311333333333, 1570.9296666666667])
  [1] 5
mag3hz_2022-04-08_15-31-11:
  [0] ndarray(shape=(2,), dtype=float64, values=[1610.2425, 1690.5783])
  [1] 3

SPIKES branch summary
Numeric arrays scanned: 560
Largest spike time across spikes branch: 2179.4254666666666


In [55]:
# Export per-file / per-experiment summary CSV for all entries in File_names
# time_windows column: JSON-encoded list of [t_onset, t_offset] pairs (always an array of arrays)
import os
import csv
import json
import pickle
from pathlib import Path
import numpy as np

if 'File_names' not in globals():
    raise RuntimeError('File_names is not defined in the current kernel.')

candidate_roots = []
for var_name in ['direct_path', 'module_path', 'data_file_path']:
    val = globals().get(var_name, None)
    if isinstance(val, str) and val:
        candidate_roots.append(val)

candidate_roots.extend([
    os.getcwd(),
    '/Users/markusmeister/Work/Grants/MagnetSearch/Analysis MM',
    '/Users/markusmeister/Work/Grants/MagnetSearch/Github/MagnetSearch',
    '/Users/markusmeister/Work/Grants/MagnetSearch',
])

seen_roots = set()
roots = []
for r in candidate_roots:
    if r not in seen_roots:
        seen_roots.add(r)
        roots.append(r)


def resolve_file_path(fn):
    candidates = []
    if 'file_aux_keys_summary' in globals() and fn in file_aux_keys_summary:
        p0 = file_aux_keys_summary[fn].get('path_used', None)
        if p0:
            candidates.insert(0, p0)
    if os.path.isabs(fn):
        candidates.append(fn)
    else:
        candidates.append(fn)
        for root in roots:
            candidates.append(os.path.join(root, fn))
    base = os.path.basename(fn)
    for root in roots:
        rp = Path(root)
        if rp.exists() and rp.is_dir():
            try:
                for p in rp.rglob(base):
                    candidates.append(str(p))
            except Exception:
                pass
    seen = set()
    for p in candidates:
        if p not in seen:
            seen.add(p)
            if os.path.exists(p):
                return p
    return None


def load_data(path):
    lower = path.lower()
    if lower.endswith('.pickle') or lower.endswith('.pkl'):
        with open(path, 'rb') as f:
            return pickle.load(f)
    obj = np.load(path, allow_pickle=True)
    if hasattr(obj, 'item'):
        try:
            return obj.item()
        except Exception:
            return obj
    return obj


def extract_time_windows(tw):
    """Convert any aux time-window format to a list of [t_onset, t_offset] pairs."""
    if isinstance(tw, np.ndarray):
        if tw.ndim == 1 and tw.shape[0] == 2:
            return [[float(tw[0]), float(tw[1])]]
        if tw.ndim == 2 and tw.shape[1] == 2:
            return [[float(r[0]), float(r[1])] for r in tw]
    if isinstance(tw, (list, tuple)):
        # singleton wrap: [[a, b]]
        if len(tw) == 1 and isinstance(tw[0], (list, tuple, np.ndarray)):
            inner = tw[0]
            if isinstance(inner, np.ndarray):
                inner = inner.tolist()
            return [[float(inner[0]), float(inner[1])]]
        # flat pair: [a, b]
        if len(tw) == 2:
            try:
                return [[float(tw[0]), float(tw[1])]]
            except (TypeError, ValueError):
                pass
        # nested list: [[a1, b1], [a2, b2], ...]
        result = []
        for item in tw:
            if isinstance(item, (list, tuple, np.ndarray)):
                if isinstance(item, np.ndarray):
                    item = item.tolist()
                result.append([float(item[0]), float(item[1])])
        if result:
            return result
    return None


def parse_aux_entry(aux_val):
    time_windows = None
    frequency = None
    if isinstance(aux_val, (list, tuple)) and len(aux_val) >= 2:
        time_windows = extract_time_windows(aux_val[0])
        try:
            frequency = float(aux_val[1])
        except Exception:
            pass
    return time_windows, frequency


rows = []
failed_files = []

for fn in File_names:
    path = resolve_file_path(fn)
    if path is None:
        failed_files.append((fn, 'file not found'))
        continue

    try:
        data_i = load_data(path)
    except Exception as e:
        failed_files.append((fn, f'load error: {type(e).__name__}: {e}'))
        continue

    spikes_branch = data_i.get('spikes', {}) if isinstance(data_i, dict) else {}
    aux_branch = data_i.get('aux', {}) if isinstance(data_i, dict) else {}
    neuron_count = len(spikes_branch.keys()) if isinstance(spikes_branch, dict) else 0

    if isinstance(aux_branch, dict) and len(aux_branch) > 0:
        for exp_name, exp_val in aux_branch.items():
            time_windows, frequency = parse_aux_entry(exp_val)
            rows.append({
                'file_name': fn,
                'n_neurons': neuron_count,
                'experiment_name': exp_name,
                'time_windows': json.dumps(time_windows) if time_windows is not None else None,
                'frequency': frequency,
            })
    else:
        rows.append({
            'file_name': fn,
            'n_neurons': neuron_count,
            'experiment_name': None,
            'time_windows': None,
            'frequency': None,
        })

out_csv = os.path.join(os.getcwd(), 'file_names_experiment_summary.csv')

with open(out_csv, 'w', newline='') as f:
    writer = csv.DictWriter(
        f,
        fieldnames=['file_name', 'n_neurons', 'experiment_name', 'time_windows', 'frequency']
    )
    writer.writeheader()
    writer.writerows(rows)

print(f'Wrote CSV: {out_csv}')
print(f'Total rows written: {len(rows)}')
print(f'Files processed: {len(File_names) - len(failed_files)} / {len(File_names)}')

if failed_files:
    print('\nFiles with errors:')
    for fn, msg in failed_files:
        print(f'  {fn}: {msg}')

print('\nFirst 10 rows:')
for r in rows[:10]:
    tw = r['time_windows']
    preview = tw[:80] + '...' if tw and len(tw) > 80 else tw
    print({**r, 'time_windows': preview})

Wrote CSV: c:\Users\dan\Documents\magnet_search\file_names_experiment_summary.csv
Total rows written: 156
Files processed: 14 / 16

Files with errors:
  MM_20211203.pickle: file not found
  MM_20211123.pickle: file not found

First 10 rows:
{'file_name': 'MM_20220228_secondsite.pickle', 'n_neurons': 140, 'experiment_name': '2nd site 2 Hz_2022-02-28_17-48-17', 'time_windows': '[[134.90673333333334, 223.40586666666667]]', 'frequency': 2.0}
{'file_name': 'MM_20220228_secondsite.pickle', 'n_neurons': 140, 'experiment_name': '2nd site 5 Hz_2022-02-28_17-50-52', 'time_windows': '[[278.9171666666667, 343.91706666666664]]', 'frequency': 5.0}
{'file_name': 'MM_20220228_secondsite.pickle', 'n_neurons': 140, 'experiment_name': '2nd site 10 Hz_2022-02-28_17-53-18', 'time_windows': '[[381.383, 414.38196666666664]]', 'frequency': 10.0}
{'file_name': 'MM_20220228_secondsite.pickle', 'n_neurons': 140, 'experiment_name': '2nd site 1 Hz_2022-02-28_17-54-46', 'time_windows': '[[446.5148, 524.515033333333

In [56]:
# Add metadata from species_region_data.py to the experiment summary CSV
# Columns added: date, subject, region, species, experiment_type
import os
import re
import csv

spec_ns = {}
with open(os.path.join(os.getcwd(), 'species_region_data.py')) as f:
    exec(f.read(), spec_ns)
annot_d = spec_ns['annot_d']

def lookup_annot(name):
    """Look up experiment name in annot_d, trying a normalised key if exact match fails.
    The pickle keys omit the underscore before the orientation angle (e.g. '3Hz0' vs '3Hz_0')."""
    info = annot_d.get(name)
    if info is None:
        info = annot_d.get(re.sub(r'(Hz)(\d+)$', r'\1_\2', name))
    return info

csv_path = os.path.join(os.getcwd(), 'file_names_experiment_summary.csv')

with open(csv_path, newline='') as f:
    rows = list(csv.DictReader(f))

n_matched = 0
unmatched = []

for row in rows:
    info = lookup_annot(row.get('experiment_name') or '')
    if info is not None:
        row['date'], row['subject'], row['region'], row['species'], row['experiment_type'] = info
        row["date"] = row["date"].split("_")
        n_matched += 1
    else:
        row['date'] = row['subject'] = row['region'] = row['species'] = row['experiment_type'] = None
        unmatched.append(row['experiment_name'])

fieldnames = [
    'file_name', 'n_neurons', 'experiment_name', 'time_windows', 'frequency',
    'date', 'subject', 'region', 'species', 'experiment_type',
]

with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f'Wrote {csv_path}')
print(f'Matched: {n_matched} / {len(rows)}')
if unmatched:
    print(f'\nUnmatched ({len(unmatched)}):')
    for name in unmatched:
        print(f'  {name!r}')

Wrote c:\Users\dan\Documents\magnet_search\file_names_experiment_summary.csv
Matched: 156 / 156


In [57]:
# Detect aux-format variations in time-window structure
import os
import pickle
import numpy as np
from pathlib import Path

candidate_roots = []
for var_name in ['direct_path', 'module_path', 'data_file_path']:
    val = globals().get(var_name, None)
    if isinstance(val, str) and val:
        candidate_roots.append(val)
candidate_roots.extend([
    os.getcwd(),
    '/Users/markusmeister/Work/Grants/MagnetSearch/Analysis MM',
    '/Users/markusmeister/Work/Grants/MagnetSearch/Github/MagnetSearch',
    '/Users/markusmeister/Work/Grants/MagnetSearch',
])
roots = list(dict.fromkeys(candidate_roots))


def resolve_file_path(fn):
    cands = []
    if 'file_aux_keys_summary' in globals() and fn in file_aux_keys_summary:
        p0 = file_aux_keys_summary[fn].get('path_used', None)
        if p0:
            cands.append(p0)
    if os.path.isabs(fn):
        cands.append(fn)
    else:
        cands.append(fn)
        for r in roots:
            cands.append(os.path.join(r, fn))

    base = os.path.basename(fn)
    for r in roots:
        rp = Path(r)
        if rp.exists() and rp.is_dir():
            try:
                for p in rp.rglob(base):
                    cands.append(str(p))
            except Exception:
                pass

    for p in dict.fromkeys(cands):
        if os.path.exists(p):
            return p
    return None


def load_data(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


variations = []
for fn in File_names:
    path = resolve_file_path(fn)
    if path is None:
        continue
    try:
        d = load_data(path)
    except Exception:
        continue

    aux = d.get('aux', {}) if isinstance(d, dict) else {}
    if not isinstance(aux, dict):
        continue

    for exp_name, exp_val in aux.items():
        if not (isinstance(exp_val, (list, tuple)) and len(exp_val) >= 2):
            continue
        tw = exp_val[0]
        if isinstance(tw, np.ndarray):
            tw = tw.tolist()

        # Flag nested/singleton wrapping or non-numeric pair shape
        is_variation = False
        reason = None

        if isinstance(tw, (list, tuple)) and len(tw) == 1 and isinstance(tw[0], (list, tuple, np.ndarray)):
            is_variation = True
            reason = 'singleton wrapper around time pair'
        elif isinstance(tw, (list, tuple)) and len(tw) >= 2 and isinstance(tw[0], (list, tuple, np.ndarray)):
            is_variation = True
            reason = 'nested list in first element'

        if is_variation:
            variations.append({
                'file_name': fn,
                'experiment_name': exp_name,
                'raw_time_window': repr(tw),
                'reason': reason,
            })

print(f'Found {len(variations)} aux-format variations.')
for v in variations[:10]:
    print(v)


Found 20 aux-format variations.
{'file_name': 'MM_20230415.pickle', 'experiment_name': '2023-04-15_16-37-23_W1R_3D_WN_Samechan', 'raw_time_window': '[[2267.428430573885, 2567.428430573885]]', 'reason': 'singleton wrapper around time pair'}
{'file_name': 'MM_20230414_firstsite.pickle', 'experiment_name': '2023-04-14_14-56-27_W25R_WN_IndepCh', 'raw_time_window': '[[1423.3283438712256, 1723.3283438712256]]', 'reason': 'singleton wrapper around time pair'}
{'file_name': 'MM_20230414_firstsite.pickle', 'experiment_name': '2023-04-14_15-01-52_W25R_WN_SameCh', 'raw_time_window': '[[1748.4448406118775, 2048.4448406118777]]', 'reason': 'singleton wrapper around time pair'}
{'file_name': 'MM_20230413_secondsite.pickle', 'experiment_name': '2023-04-13_17-21-57_W25R_second_site_WN_SamCh', 'raw_time_window': '[[844.116630787176, 1144.1166307871758]]', 'reason': 'singleton wrapper around time pair'}
{'file_name': 'MM_20230413_secondsite.pickle', 'experiment_name': '2023-04-13_17-42-25_W25R_second_si

In [58]:
# Compact report of detected aux-format variations
if 'variations' not in globals():
    print('No variations variable found. Run detection cell first.')
else:
    print('n_variations =', len(variations))
    if len(variations) > 0:
        v = variations[0]
        raw = v['raw_time_window']
        if len(raw) > 200:
            raw = raw[:200] + '...'
        print('example_file =', v['file_name'])
        print('example_experiment =', v['experiment_name'])
        print('example_reason =', v['reason'])
        print('example_raw_time_window =', raw)

n_variations = 20
example_file = MM_20230415.pickle
example_experiment = 2023-04-15_16-37-23_W1R_3D_WN_Samechan
example_reason = singleton wrapper around time pair
example_raw_time_window = [[2267.428430573885, 2567.428430573885]]
